# MEI generation for 3D video model — val/test-ensemble protocol

Extends `mei_video_model.ipynb` with **three disjoint model ensembles** (same protocol as `mei_ensemble_valtest_atfirstsight.ipynb`):

| ensemble | role | seeds (default) |
|----------|------|------------------|
| **MEI**  | drives gradient ascent (objective ensemble) | 42, 43, 44 |
| **VAL**  | held-out signal for early stopping          | 52, 53     |
| **TEST** | unbiased final activation report            | 62         |

**Per neuron**:
1. Gradient ascent on MEI-ensemble mean activation (Fourier preconditioning, random jitter, PNorm constraint).
2. After every step, evaluate val-ensemble mean activation — keep a running *best-val snapshot*.
3. If val activation hasn’t improved for `patience` steps (after `warmup`), stop early and return the best-val snapshot.
4. Report the test-ensemble mean activation on that snapshot — the unbiased number.

**Two MEI types**:
- **Static MEI**: optimize a single frame `(1, 1, H, W)` repeated T times — spatial RF only
- **Video MEI**: optimize a full spatio-temporal clip `(1, 1, T, H, W)` — spatial + temporal RF

**Key design decisions** (from `MEI_OPTIMIZATION_REPORT.md`):
- No Gaussian blur — causes energy drain with the 3D core’s weak gradients
- Fourier preconditioning (alpha=0.6) controls smoothness on the gradient side instead
- RandomJitter (±2px) prevents pixel-aligned artifacts

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
import types
import pathlib

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from tqdm.notebook import tqdm

# Mock datajoint and gitpython (only needed by nnfabrik's dj_helpers, not used here)
_dj = types.ModuleType('datajoint')
_dj.config = {}
for sub in ['utils', 'schema', 'schemas', 'fetch']:
    mod = types.ModuleType(f'datajoint.{sub}')
    setattr(_dj, sub, mod)
    sys.modules[f'datajoint.{sub}'] = mod
_dj.utils.to_camel_case = lambda s: s
_dj.schema.Schema = type('Schema', (), {})
_dj.schemas.Schema = type('Schema', (), {})
_dj.fetch.DataJointError = Exception
sys.modules['datajoint'] = _dj

_git = types.ModuleType('git')
_git.Repo = type('Repo', (), {})
_git.cmd = types.ModuleType('git.cmd')
sys.modules['git'] = _git
sys.modules['git.cmd'] = _git.cmd

# Add package paths
BASE = '/mnt/vast-nhr/projects/nix00014/goirik/MOZAIK-new'
sys.path.insert(0, f'{BASE}/sensorium/notebooks/model_tutorial/nnfabrik_base')
sys.path.insert(0, f'{BASE}/experanto')
sys.path.insert(0, f'{BASE}/mei')
sys.path.insert(0, f'{BASE}/neuralpredictors')
sys.path.insert(0, f'{BASE}/latent_space_model')

from sensorium.models.make_model import make_video_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

## 1. Configure the three ensembles

All members must share the same model architecture and data key. Train multiple seeds by
re-running your training script with different `seed` values and saving separate checkpoints.

Update `CKPT_TEMPLATE` and seed lists to match your checkpoint naming.

In [ ]:
# --- Checkpoint template ---
# Adjust this to match your naming convention
CKPT_TEMPLATE = '../../test_training/single_session_26872_no_beh_seed{seed}/best.pth'

MEI_SEEDS  = [42, 43, 44]
VAL_SEEDS  = [52, 53]
TEST_SEEDS = [62]

def ckpts_for(seeds):
    return [CKPT_TEMPLATE.format(seed=s) for s in seeds]

CHECKPOINTS_MEI  = ckpts_for(MEI_SEEDS)
CHECKPOINTS_VAL  = ckpts_for(VAL_SEEDS)
CHECKPOINTS_TEST = ckpts_for(TEST_SEEDS)

for label, lst in [('MEI', CHECKPOINTS_MEI), ('VAL', CHECKPOINTS_VAL), ('TEST', CHECKPOINTS_TEST)]:
    print(f'{label}: {len(lst)} model(s)')
    for p in lst:
        present = '\u2713' if pathlib.Path(p).is_file() else '\u2717 MISSING'
        print(f'  {present}  {p}')

In [ ]:
# --- Model architecture config (must match training exactly) ---
factorised_3D_core_dict = dict(
    input_channels=1,
    hidden_channels=[32, 64, 128],
    spatial_input_kernel=(11, 11),
    temporal_input_kernel=11,
    spatial_hidden_kernel=(5, 5),
    temporal_hidden_kernel=5,
    stride=1,
    layers=3,
    gamma_input_spatial=10,
    gamma_input_temporal=0.01,
    bias=True,
    hidden_nonlinearities='elu',
    x_shift=0,
    y_shift=0,
    batch_norm=True,
    laplace_padding=None,
    input_regularizer='LaplaceL2norm',
    padding=False,
    final_nonlin=True,
    momentum=0.7,
)

readout_dict = dict(
    bias=True,
    init_mu_range=0.2,
    init_sigma=1.0,
    gamma_readout=0.0,
    gauss_type='full',
    grid_mean_predictor=None,
    share_features=False,
    share_grid=False,
    shared_match_ids=None,
    gamma_grid_dispersion=0.0,
    zig=False,
    out_channels=1,
    kernel_size=(11, 5),
    batch_size=4,
)

# Spatial dimensions at scale=0.25
H, W = 36, 64

## 2. Load all ensembles

In [ ]:
def load_video_model(ckpt_path, seed=42):
    """Load a VideoFiringRateEncoder from checkpoint without dataloaders."""
    state_dict = torch.load(ckpt_path, map_location='cpu')

    # Extract data_key and n_neurons from checkpoint
    readout_keys = [k for k in state_dict.keys() if k.startswith('readout.')]
    data_key = readout_keys[0].split('.')[1]
    n_neurons = state_dict[f'readout.{data_key}.bias'].shape[0]

    n_neurons_dict = {data_key: n_neurons}
    mean_activity_dict = {data_key: torch.zeros(n_neurons)}

    model = make_video_model(
        None, seed,
        core_dict=factorised_3D_core_dict,
        core_type='3D_factorised',
        readout_dict=readout_dict.copy(),
        readout_type='gaussian',
        use_gru=False, gru_dict=None,
        use_shifter=False, shifter_dict=None, shifter_type=None,
        deeplake_ds=False,
        n_neurons_dict=n_neurons_dict,
        mean_activity_dict=mean_activity_dict,
        experanto=True,
        readout_dim=factorised_3D_core_dict['hidden_channels'][-1],
    )
    model.load_state_dict(state_dict)
    model.to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, data_key, n_neurons


def load_ensemble(ckpt_paths, base_seed=42):
    models, data_keys, neuron_counts = [], [], []
    for i, path in enumerate(ckpt_paths):
        m, dk, nn_ = load_video_model(path, seed=base_seed + i)
        models.append(m)
        data_keys.append(dk)
        neuron_counts.append(nn_)
    # Sanity: all must share data_key and n_neurons
    assert len(set(data_keys)) == 1, f'Mismatched data_keys: {data_keys}'
    assert len(set(neuron_counts)) == 1, f'Mismatched n_neurons: {neuron_counts}'
    return models, data_keys[0], neuron_counts[0]


ensemble_mei, DATA_KEY, N_NEURONS = load_ensemble(CHECKPOINTS_MEI, base_seed=42)
ensemble_val, _, _ = load_ensemble(CHECKPOINTS_VAL, base_seed=52)
ensemble_test, _, _ = load_ensemble(CHECKPOINTS_TEST, base_seed=62)

print(f'Data key: {DATA_KEY}, Neurons: {N_NEURONS}')
print(f'Sizes: MEI={len(ensemble_mei)}  VAL={len(ensemble_val)}  TEST={len(ensemble_test)}')

In [ ]:
# --- Determine temporal shrinkage ---
with torch.no_grad():
    test_T = 50
    test_input = torch.randn(1, 1, test_T, H, W).to(device)
    test_output = ensemble_mei[0](test_input, data_key=DATA_KEY)
    T_OUT = test_output.shape[1]
    TEMPORAL_LOSS = test_T - T_OUT
    print(f'Input T={test_T} -> Output T\'={T_OUT}, temporal loss={TEMPORAL_LOSS} frames')
    print(f'Output shape: {test_output.shape}')  # (1, T', n_neurons)

VIDEO_T = 50
print(f'\nVideo MEI will optimize (1, 1, {VIDEO_T}, {H}, {W}) -> output T\'={VIDEO_T - TEMPORAL_LOSS}')

## 3. Rank neurons by MEI-ensemble test correlation

If you have dataloaders, rank neurons by the MEI ensemble's test-tier correlation.
Otherwise, fall back to linearly spaced indices.

In [ ]:
# --- Option A: rank by correlation (requires dataloaders) ---
# Uncomment and adapt if you have video dataloaders:
#
# from sensorium.datasets.mouse_video_loaders import mouse_video_loader
# from sensorium.utility.scores import get_correlations
#
# dataloaders = mouse_video_loader(
#     paths=['path/to/your/data'],
#     batch_size=8,
#     scale=0.25,
#     frames=VIDEO_T,
# )
# per_model_corr = [
#     get_correlations(m, dataloaders, tier='test', device=device, as_dict=False, per_neuron=True)
#     for m in ensemble_mei
# ]
# mei_corr = np.mean(per_model_corr, axis=0)
# ranked_neurons = np.argsort(mei_corr)[::-1]

# --- Option B: use linearly spaced neurons (no dataloaders needed) ---
mei_corr = None  # set to correlation array if available
ranked_neurons = np.linspace(0, N_NEURONS - 1, N_NEURONS, dtype=int)  # all neurons, unranked

print(f'n_neurons: {N_NEURONS}')
if mei_corr is not None:
    print(f'top-5 rho (MEI ensemble): {mei_corr[ranked_neurons[:5]].round(3)}')
    print(f'median rho: {np.median(mei_corr):.3f}')
else:
    print('No correlation ranking — using linear neuron ordering')

## 4. Regularization callables

- **Fourier preconditioning** (alpha=0.6): amplifies low-frequency gradient components
- **RandomJitter** (±2px): prevents pixel-aligned artifacts
- **PNormConstraintAndClip**: L1 norm + pixel clipping

No Gaussian blur — it causes energy drain with the 3D core's weak gradients (see report §3.4).

Both 2D (static MEI) and video (5D) versions are provided.

In [ ]:
# --- 2D versions (for static MEI, input shape (B, C, H, W)) ---

class FourierPrecondition:
    """Lowpass-filter gradient in Fourier space: G(fx,fy) = (fx^2+fy^2)^(-alpha)."""
    def __init__(self, alpha=0.6):
        self.alpha = alpha
        self._filter_cache = {}

    def _get_filter(self, H, W, device):
        key = (H, W, device)
        if key not in self._filter_cache:
            fy = torch.fft.fftfreq(H, device=device).unsqueeze(1)
            fx = torch.fft.fftfreq(W, device=device).unsqueeze(0)
            freq_sq = fx ** 2 + fy ** 2
            freq_sq[0, 0] = 1.0
            filt = freq_sq ** (-self.alpha)
            filt[0, 0] = 1.0
            self._filter_cache[key] = filt
        return self._filter_cache[key]

    def __call__(self, grad, i_iteration):
        H, W = grad.shape[-2:]
        filt = self._get_filter(H, W, grad.device)
        return torch.fft.ifft2(torch.fft.fft2(grad) * filt).real


class RandomJitter:
    """Random spatial jitter for 2D input (B, C, H, W)."""
    def __init__(self, amount=2):
        self.amount = amount

    def __call__(self, mei, i_iteration):
        ox = torch.randint(-self.amount, self.amount + 1, (1,)).item()
        oy = torch.randint(-self.amount, self.amount + 1, (1,)).item()
        return torch.roll(mei, shifts=(ox, oy), dims=(2, 3))


# --- Video versions (for video MEI, input shape (B, C, T, H, W)) ---

class FourierPreconditionVideo:
    """Lowpass-filter gradient per frame in Fourier space."""
    def __init__(self, alpha=0.6):
        self.alpha = alpha
        self._filter_cache = {}

    def _get_filter(self, H, W, device):
        key = (H, W, device)
        if key not in self._filter_cache:
            fy = torch.fft.fftfreq(H, device=device).unsqueeze(1)
            fx = torch.fft.fftfreq(W, device=device).unsqueeze(0)
            freq_sq = fx ** 2 + fy ** 2
            freq_sq[0, 0] = 1.0
            filt = freq_sq ** (-self.alpha)
            filt[0, 0] = 1.0
            self._filter_cache[key] = filt
        return self._filter_cache[key]

    def __call__(self, grad, i_iteration):
        H, W = grad.shape[-2:]
        filt = self._get_filter(H, W, grad.device)
        return torch.fft.ifft2(torch.fft.fft2(grad) * filt).real


class RandomJitterVideo:
    """Random spatial jitter for video (B, C, T, H, W). Rolls spatial dims 3, 4."""
    def __init__(self, amount=2):
        self.amount = amount

    def __call__(self, mei, i_iteration):
        ox = torch.randint(-self.amount, self.amount + 1, (1,)).item()
        oy = torch.randint(-self.amount, self.amount + 1, (1,)).item()
        return torch.roll(mei, shifts=(ox, oy), dims=(3, 4))

## 5. MEI generator with val-driven early stopping

Supports both **static** (`mode='static'`) and **video** (`mode='video'`) MEI types.

- Gradient ascent driven by MEI-ensemble mean activation
- Val-ensemble tracks best snapshot (after warmup)
- Test-ensemble logged for unbiased evaluation
- Fourier preconditioning + random jitter (paper recipe, no Gaussian blur)
- PNormConstraintAndClip for energy constraint

In [ ]:
def ensemble_mean_activation(models, x, neuron_idx, data_key):
    """Mean activation of neuron_idx across model ensemble.
    x: (1, 1, T, H, W) for video or (1, 1, H, W) expanded to video internally.
    Returns scalar tensor with grad.
    """
    acts = []
    for m in models:
        output = m(x, data_key=data_key)  # (1, T', n_neurons)
        acts.append(output[:, :, neuron_idx].mean())  # mean over time
    return torch.stack(acts).mean()


def generate_mei_valtest(
    mei_models, val_models, test_models,
    data_key, neuron_idx,
    mode='static',
    T=50, H=36, W=64,
    n_steps=500, lr=0.1,
    precond_alpha=0.6,
    jitter_amount=2,
    norm_value=30.0, p_norm=1,
    clamp_min=-1.0, clamp_max=1.0,
    patience=150, warmup=50,
    seed=42, device='cuda', verbose=False,
):
    """MEI optimization with val-driven early stopping for 3D video model.

    Args:
        mode: 'static' optimizes (1,1,H,W) repeated T times; 'video' optimizes (1,1,T,H,W).

    Returns:
        (best_img_or_video_np, info)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    if mode == 'static':
        img = torch.randn(1, 1, H, W, device=device) * 0.1
        preconditioner = FourierPrecondition(alpha=precond_alpha)
        jitter = RandomJitter(amount=jitter_amount)
        effective_norm = norm_value
    elif mode == 'video':
        img = torch.randn(1, 1, T, H, W, device=device) * 0.1
        preconditioner = FourierPreconditionVideo(alpha=precond_alpha)
        jitter = RandomJitterVideo(amount=jitter_amount)
        effective_norm = norm_value * (T ** 0.5)  # scale norm with temporal extent
    else:
        raise ValueError(f'Unknown mode: {mode}')

    img.requires_grad_(True)
    optimizer = torch.optim.SGD([img], lr=lr)

    best_val_act = -float('inf')
    best_step = warmup
    best_img = img.detach().clone()
    plateau = 0
    stopped_at = n_steps

    mei_trace, val_trace, test_trace = [], [], []

    iterator = tqdm(range(n_steps), desc=f'neuron {neuron_idx}', leave=False) if verbose else range(n_steps)
    for step in iterator:
        optimizer.zero_grad()

        # Transform: jitter for evaluation only
        x_eval = jitter(img, step)

        # For static mode, expand single frame to video
        if mode == 'static':
            x_video = x_eval.unsqueeze(2).expand(-1, -1, T, -1, -1)
        else:
            x_video = x_eval

        # MEI ensemble - drives the gradient
        mei_act = ensemble_mean_activation(mei_models, x_video, neuron_idx, data_key)
        (-mei_act).backward()  # negate for gradient ascent
        mei_trace.append(float(mei_act.item()))

        # Val + test ensembles - observation only
        with torch.no_grad():
            # Use un-jittered image for val/test to reduce noise
            if mode == 'static':
                x_clean = img.unsqueeze(2).expand(-1, -1, T, -1, -1)
            else:
                x_clean = img
            val_act = float(ensemble_mean_activation(val_models, x_clean, neuron_idx, data_key).item())
            test_act = float(ensemble_mean_activation(test_models, x_clean, neuron_idx, data_key).item())
        val_trace.append(val_act)
        test_trace.append(test_act)

        # Precondition gradients (Fourier lowpass)
        img.grad.data = preconditioner(img.grad.data, step)

        # Optimizer step
        optimizer.step()

        # Postprocess: PNorm constraint + clip
        with torch.no_grad():
            current_norm = img.data.abs().sum()  # L1 norm
            if current_norm > effective_norm:
                img.data.mul_(effective_norm / current_norm)
            img.data.clamp_(clamp_min, clamp_max)

        # Update best-val snapshot (skip warmup to avoid random-init spike)
        if step >= warmup:
            if val_act > best_val_act:
                best_val_act = val_act
                best_step = step
                best_img = img.detach().clone()
                plateau = 0
            else:
                plateau += 1

        # Early stopping
        if step >= warmup and plateau >= patience:
            stopped_at = step + 1
            break

    info = {
        'mei_trace':     np.asarray(mei_trace, dtype=np.float32),
        'val_trace':     np.asarray(val_trace, dtype=np.float32),
        'test_trace':    np.asarray(test_trace, dtype=np.float32),
        'best_step':     int(best_step),
        'stopped_at':    int(stopped_at),
        'best_val_act':  float(best_val_act),
        'best_test_act': float(test_trace[best_step]),
        'best_mei_act':  float(mei_trace[best_step]),
        'mode':          mode,
    }
    return best_img.detach().cpu().numpy(), info

## 6. Generate static MEIs for top-K neurons

In [ ]:
N_TO_GENERATE = 64
N_STEPS       = 500
PATIENCE      = 150
WARMUP        = 50

top_ids = ranked_neurons[:N_TO_GENERATE]

static_meis, static_infos = {}, {}
for nid in tqdm(top_ids, desc='static MEIs'):
    mei, info = generate_mei_valtest(
        ensemble_mei, ensemble_val, ensemble_test,
        DATA_KEY, int(nid),
        mode='static', T=VIDEO_T, H=H, W=W,
        n_steps=N_STEPS, lr=0.1,
        patience=PATIENCE, warmup=WARMUP,
        seed=42, device=device,
    )
    static_meis[int(nid)] = mei
    static_infos[int(nid)] = info

stop_steps = np.array([static_infos[int(i)]['stopped_at'] for i in top_ids])
best_steps = np.array([static_infos[int(i)]['best_step']  for i in top_ids])
print(f'stopped_at -- median {int(np.median(stop_steps))}, range [{stop_steps.min()}, {stop_steps.max()}]')
print(f'best_step  -- median {int(np.median(best_steps))}, range [{best_steps.min()}, {best_steps.max()}]')

## 7. Generate video MEIs for a subset of neurons

Video MEIs are more expensive (5D optimization). Generate for a smaller subset.

In [ ]:
N_VIDEO = 16  # smaller subset for video MEIs
video_ids = top_ids[:N_VIDEO]

video_meis, video_infos = {}, {}
for nid in tqdm(video_ids, desc='video MEIs'):
    mei, info = generate_mei_valtest(
        ensemble_mei, ensemble_val, ensemble_test,
        DATA_KEY, int(nid),
        mode='video', T=VIDEO_T, H=H, W=W,
        n_steps=N_STEPS, lr=0.1,
        patience=PATIENCE, warmup=WARMUP,
        seed=42, device=device,
    )
    video_meis[int(nid)] = mei
    video_infos[int(nid)] = info

v_stop = np.array([video_infos[int(i)]['stopped_at'] for i in video_ids])
v_best = np.array([video_infos[int(i)]['best_step']  for i in video_ids])
print(f'Video MEIs: stopped_at -- median {int(np.median(v_stop))}, range [{v_stop.min()}, {v_stop.max()}]')
print(f'Video MEIs: best_step  -- median {int(np.median(v_best))}, range [{v_best.min()}, {v_best.max()}]')

## 8. Save results

In [ ]:
out_dir = pathlib.Path('../../figures/mei/video_model_valtest')
out_dir.mkdir(parents=True, exist_ok=True)

# Save static MEIs
save_kwargs = {f'neuron_{k}': v for k, v in static_meis.items()}
for k, info in static_infos.items():
    save_kwargs[f'neuron_{k}__mei_trace']  = info['mei_trace']
    save_kwargs[f'neuron_{k}__val_trace']  = info['val_trace']
    save_kwargs[f'neuron_{k}__test_trace'] = info['test_trace']
np.savez(
    out_dir / 'static_meis_valtest_ensemble.npz',
    __ranked_ids=np.array(top_ids),
    __best_steps=np.array([static_infos[int(i)]['best_step'] for i in top_ids]),
    __stopped_at=np.array([static_infos[int(i)]['stopped_at'] for i in top_ids]),
    __best_val_act=np.array([static_infos[int(i)]['best_val_act'] for i in top_ids], dtype=np.float32),
    __best_test_act=np.array([static_infos[int(i)]['best_test_act'] for i in top_ids], dtype=np.float32),
    __best_mei_act=np.array([static_infos[int(i)]['best_mei_act'] for i in top_ids], dtype=np.float32),
    **save_kwargs,
)
print(f'Saved {len(static_meis)} static MEIs -> {out_dir / "static_meis_valtest_ensemble.npz"}')

# Save video MEIs
save_kwargs_v = {f'neuron_{k}': v for k, v in video_meis.items()}
for k, info in video_infos.items():
    save_kwargs_v[f'neuron_{k}__mei_trace']  = info['mei_trace']
    save_kwargs_v[f'neuron_{k}__val_trace']  = info['val_trace']
    save_kwargs_v[f'neuron_{k}__test_trace'] = info['test_trace']
np.savez(
    out_dir / 'video_meis_valtest_ensemble.npz',
    __ranked_ids=np.array(list(video_ids)),
    __best_steps=np.array([video_infos[int(i)]['best_step'] for i in video_ids]),
    __stopped_at=np.array([video_infos[int(i)]['stopped_at'] for i in video_ids]),
    __best_val_act=np.array([video_infos[int(i)]['best_val_act'] for i in video_ids], dtype=np.float32),
    __best_test_act=np.array([video_infos[int(i)]['best_test_act'] for i in video_ids], dtype=np.float32),
    __best_mei_act=np.array([video_infos[int(i)]['best_mei_act'] for i in video_ids], dtype=np.float32),
    **save_kwargs_v,
)
print(f'Saved {len(video_meis)} video MEIs -> {out_dir / "video_meis_valtest_ensemble.npz"}')

## 9. Trace diagnostics

Plot per-step activation for each ensemble alongside the val-best snapshot and early-stop point.

In [ ]:
def plot_traces(infos_dict, neuron_ids, warmup, title_prefix='', corr_arr=None):
    N_TRACES = min(6, len(neuron_ids))
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=False)
    axes = axes.ravel()
    for ax, nid in zip(axes, neuron_ids[:N_TRACES]):
        info = infos_dict[int(nid)]
        steps = np.arange(len(info['mei_trace']))
        ax.axvspan(0, warmup, color='gray', alpha=0.12)
        ax.plot(steps, info['mei_trace'],  label='MEI ens',  color='C0', lw=1.4)
        ax.plot(steps, info['val_trace'],  label='val ens',  color='C1', lw=1.4)
        ax.plot(steps, info['test_trace'], label='test ens', color='C2', lw=1.4)
        ax.axvline(info['best_step'],  color='C1', ls='--', alpha=0.6, label='best val')
        ax.axvline(info['stopped_at'], color='k',  ls=':',  alpha=0.5, label='stopped')
        rho_str = f'  rho={corr_arr[int(nid)]:.2f}' if corr_arr is not None else ''
        ax.set_title(f'neuron {int(nid)}{rho_str}', fontsize=10)
        ax.set_xlabel('step'); ax.set_ylabel('activation')
        ax.legend(fontsize=7, loc='lower right')
    # Hide unused axes
    for ax in axes[N_TRACES:]:
        ax.set_visible(False)
    plt.suptitle(f'{title_prefix}Per-step activation traces', fontsize=12)
    plt.tight_layout()
    return fig


fig_s = plot_traces(static_infos, top_ids, WARMUP, title_prefix='Static MEI: ', corr_arr=mei_corr)
fig_s.savefig(out_dir / 'static_mei_valtest_traces.png', dpi=150, bbox_inches='tight')
plt.show()

if video_infos:
    fig_v = plot_traces(video_infos, video_ids, WARMUP, title_prefix='Video MEI: ', corr_arr=mei_corr)
    fig_v.savefig(out_dir / 'video_mei_valtest_traces.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Static MEI grid (val-selected snapshots)

Per-MEI median subtraction, shared P99 colormap range.

In [ ]:
GRID_ROWS, GRID_COLS = 8, 8
n_grid = min(GRID_ROWS * GRID_COLS, len(static_meis))
ordered_ids = list(top_ids[:n_grid])

# Per-MEI median subtraction + shared P99 range
normalized = np.stack([static_meis[int(i)].squeeze() - np.median(static_meis[int(i)]) for i in ordered_ids])
vmax = np.percentile(np.abs(normalized), 99); vmin = -vmax

actual_rows = int(np.ceil(n_grid / GRID_COLS))
fig, axes = plt.subplots(actual_rows, GRID_COLS,
                         figsize=(GRID_COLS * 2.0, actual_rows * 1.5),
                         gridspec_kw=dict(hspace=0.55, wspace=0.05))
if actual_rows == 1:
    axes = axes[np.newaxis, :]
for k, nid in enumerate(ordered_ids):
    r, c = divmod(k, GRID_COLS); ax = axes[r, c]
    ax.imshow(normalized[k], cmap='gray', vmin=vmin, vmax=vmax, aspect='auto')
    info = static_infos[int(nid)]
    ax.set_title(f'{int(nid)}\n'
                 f'val={info["best_val_act"]:.2f}  test={info["best_test_act"]:.2f}',
                 fontsize=7)
    ax.axis('off')
# Hide unused axes
for k in range(n_grid, actual_rows * GRID_COLS):
    r, c = divmod(k, GRID_COLS)
    axes[r, c].set_visible(False)

plt.suptitle(f'Top {n_grid} Static MEIs -- val-selected snapshots\n'
             f'MEI ens={len(ensemble_mei)}, VAL ens={len(ensemble_val)}, TEST ens={len(ensemble_test)} '
             f'-- display +/-{vmax:.2f} (P99)', fontsize=11, y=1.005)
plt.tight_layout()
plt.savefig(out_dir / 'static_mei_valtest_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Video MEI visualization

For each video MEI: frame montage (evenly spaced) and temporal summary (mean + std).

In [ ]:
def plot_video_mei_montage(video_mei, neuron_idx, info=None, n_show=8):
    """Show evenly spaced frames from a video MEI."""
    video = video_mei.squeeze()  # (T, H, W)
    T = video.shape[0]
    frame_ids = np.linspace(0, T - 1, n_show, dtype=int)
    fig, axes = plt.subplots(1, n_show + 2, figsize=(2 * (n_show + 2), 2.5))
    for ax, fi in zip(axes[:n_show], frame_ids):
        ax.imshow(video[fi], cmap='gray', vmin=-1, vmax=1)
        ax.set_title(f't={fi}', fontsize=9)
        ax.axis('off')
    # Temporal mean
    axes[n_show].imshow(video.mean(axis=0), cmap='gray', vmin=-1, vmax=1)
    axes[n_show].set_title('mean', fontsize=9)
    axes[n_show].axis('off')
    # Temporal std
    axes[n_show + 1].imshow(video.std(axis=0), cmap='hot')
    axes[n_show + 1].set_title('std', fontsize=9)
    axes[n_show + 1].axis('off')

    title = f'Video MEI - Neuron {neuron_idx}'
    if info:
        title += f'  val={info["best_val_act"]:.2f}  test={info["best_test_act"]:.2f}'
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    return fig


for nid in video_ids[:6]:
    fig = plot_video_mei_montage(video_meis[int(nid)], int(nid), video_infos[int(nid)])
    fig.savefig(out_dir / f'video_mei_montage_neuron_{int(nid)}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 12. Static vs Video MEI comparison

Side-by-side: static MEI vs video MEI temporal mean, for neurons where both are available.

In [ ]:
common_ids = [int(nid) for nid in video_ids if int(nid) in static_meis]
n_show = min(8, len(common_ids))

fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 5))
if n_show == 1:
    axes = axes[:, np.newaxis]

for j, nid in enumerate(common_ids[:n_show]):
    # Static
    axes[0, j].imshow(static_meis[nid].squeeze(), cmap='gray', vmin=-1, vmax=1)
    s_info = static_infos[nid]
    axes[0, j].set_title(f'N{nid}\ntest={s_info["best_test_act"]:.2f}', fontsize=8)
    axes[0, j].axis('off')

    # Video temporal mean
    vmean = video_meis[nid].squeeze().mean(axis=0)
    v_info = video_infos[nid]
    axes[1, j].imshow(vmean, cmap='gray', vmin=-1, vmax=1)
    axes[1, j].set_title(f'test={v_info["best_test_act"]:.2f}', fontsize=8)
    axes[1, j].axis('off')

axes[0, 0].set_ylabel('Static MEI', fontsize=10)
axes[1, 0].set_ylabel('Video MEI\n(temporal mean)', fontsize=10)
plt.suptitle('Static vs Video MEI (val-selected, test-ensemble activation)', fontsize=12)
plt.tight_layout()
plt.savefig(out_dir / 'static_vs_video_mei_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Specificity matrix on the test ensemble

Walker et al. 2019 Fig. 2b, evaluated on the held-out test ensemble. A dominant diagonal means
MEIs generalize across model seeds.

In [ ]:
def compute_mei_responses(models, meis_dict, ordered_ids, data_key, T, H, W,
                          mode='static', device='cuda'):
    """Compute response of all neurons to each MEI via the given ensemble."""
    for m in models:
        m.eval()
    responses = []
    with torch.no_grad():
        for nid in ordered_ids:
            arr = meis_dict[int(nid)]
            if mode == 'static':
                # (H, W) or (1, 1, H, W) -> expand to video
                frame = torch.from_numpy(arr).float().view(1, 1, H, W).to(device)
                x = frame.unsqueeze(2).expand(-1, -1, T, -1, -1)
            else:
                # (1, 1, T, H, W)
                x = torch.from_numpy(arr).float().to(device)
                if x.dim() == 3:  # (T, H, W)
                    x = x.unsqueeze(0).unsqueeze(0)
                elif x.dim() == 4:  # (1, T, H, W)
                    x = x.unsqueeze(0)
            # Ensemble mean response: (T', n_neurons) -> mean over time -> (n_neurons,)
            out = torch.stack([m(x, data_key=data_key)[:, :, :].mean(dim=1)
                               for m in models]).mean(0)  # (1, n_neurons)
            responses.append(out[0].cpu().numpy())
    return np.stack(responses)  # (n_meis, n_neurons)


R_test = compute_mei_responses(ensemble_test, static_meis, ordered_ids, DATA_KEY,
                               VIDEO_T, H, W, mode='static', device=device)
sub_test = R_test[:, [int(i) for i in ordered_ids]]
col_max = sub_test.max(axis=0, keepdims=True)
col_max[col_max == 0] = 1
sub_norm = sub_test / col_max

n = sub_test.shape[0]
diag = np.diag(sub_test)
off = np.array([np.concatenate([sub_test[:i, i], sub_test[i+1:, i]]).mean() for i in range(n)])
si = diag / np.maximum(off, 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
im = axes[0].imshow(sub_norm, cmap='hot', aspect='auto')
axes[0].set_xlabel('neuron index (ranked order)')
axes[0].set_ylabel('MEI index (ranked order)')
axes[0].set_title(f'Specificity on TEST ensemble (col-normalized, n={n})')
plt.colorbar(im, ax=axes[0], shrink=0.8)

axes[1].hist(si, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(np.median(si), color='red', ls='--', label=f'median = {np.median(si):.2f}')
axes[1].set_xlabel('selectivity (own / mean others)')
axes[1].set_ylabel('# neurons')
axes[1].set_title('Selectivity index (test-ensemble)')
axes[1].legend()
plt.tight_layout()
plt.savefig(out_dir / 'static_mei_valtest_specificity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'TEST-ensemble selectivity: mean={si.mean():.2f}, median={np.median(si):.2f}')

## 14. Cross-ensemble activation comparison

For each MEI, plot its own-neuron activation as measured by the three ensembles.
Val and test should track each other; large gaps indicate MEI-ensemble overfit.

In [ ]:
best_mei  = np.array([static_infos[int(i)]['best_mei_act']  for i in ordered_ids])
best_val  = np.array([static_infos[int(i)]['best_val_act']  for i in ordered_ids])
best_test = np.array([static_infos[int(i)]['best_test_act'] for i in ordered_ids])

idx = np.arange(len(ordered_ids))
w = 0.28

fig, ax = plt.subplots(figsize=(max(8, 0.18 * len(ordered_ids)), 4))
ax.bar(idx - w, best_mei,  w, label=f'MEI ens (n={len(ensemble_mei)})',  color='C0')
ax.bar(idx,     best_val,  w, label=f'val ens (n={len(ensemble_val)})',  color='C1')
ax.bar(idx + w, best_test, w, label=f'test ens (n={len(ensemble_test)})', color='C2')
ax.set_xlabel('MEI index (ranked order)')
ax.set_ylabel('activation at val-best step')
ax.set_title('Cross-ensemble activation -- should track if MEIs generalize')
ax.legend()
ax.set_xticks(idx[::4])
plt.tight_layout()
plt.savefig(out_dir / 'static_mei_valtest_activation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

ratio_val  = best_val  / np.maximum(best_mei, 1e-8)
ratio_test = best_test / np.maximum(best_mei, 1e-8)
print(f'val/mei  ratio -- median {np.median(ratio_val):.2f}')
print(f'test/mei ratio -- median {np.median(ratio_test):.2f}')

## 15. Animate a video MEI

In [ ]:
def animate_video_mei(video_mei, neuron_idx, fps=15):
    """Create inline animation of a video MEI."""
    video = video_mei.squeeze()  # (T, H, W)
    fig, ax = plt.subplots(figsize=(5, 3))
    im = ax.imshow(video[0], cmap='gray', vmin=-1, vmax=1, animated=True)
    ax.axis('off')
    title = ax.set_title(f'Neuron {neuron_idx} - frame 0/{len(video)-1}')

    def update(frame):
        im.set_data(video[frame])
        title.set_text(f'Neuron {neuron_idx} - frame {frame}/{len(video)-1}')
        return [im, title]

    anim = animation.FuncAnimation(fig, update, frames=len(video),
                                   interval=1000 // fps, blit=True)
    plt.close(fig)
    return anim


if video_meis:
    first_vid_id = int(video_ids[0])
    anim = animate_video_mei(video_meis[first_vid_id], first_vid_id)
    HTML(anim.to_jshtml())